In [38]:
from news_data import load_saved_data
import numpy as np
data = load_saved_data("news_data.pkl")

X_train = data["X_train"]
X_test = data["X_test"]
y_train = data["y_train"]
y_test = data["y_test"]
word_to_idx = data["word_to_idx"]
vocab_size = data["vocab_size"]

1.将 X_train 和 X_test 文本 padding 到相同的长度

In [39]:
train_lengths = [len(seq) for seq in X_train]
test_lengths = [len(seq) for seq in X_test]

print("训练集样本数:", len(X_train))
print("测试集样本数:", len(X_test))
print("训练集最长长度:", max(train_lengths))
print("测试集最长长度:", max(test_lengths))
print("训练集平均长度:", sum(train_lengths) / len(train_lengths))

训练集样本数: 1079
测试集样本数: 717
训练集最长长度: 45731
测试集最长长度: 28696
训练集平均长度: 1258.7164040778498


1.1 padding function

In [40]:
def pad_sequences(sequences, max_len, pad_value=0):
    padded_sequences = []
    
    for seq in sequences:
        seq = list(seq)
        
        if len(seq) < max_len:
            seq = seq + [pad_value] * (max_len - len(seq))
        else:
            seq = seq[:max_len]
        
        padded_sequences.append(seq)
    
    return np.array(padded_sequences, dtype=np.int64)

1.2 translate from text to index

In [41]:
def texts_to_indices(texts, word_to_idx, unk_idx):
    sequences = []
    for text in texts:
        # 如果 text 是一句字符串，就按空格切分
        tokens = text.split()
        
        # 转成索引，不在词表里的词用 UNK
        seq = [word_to_idx.get(token, unk_idx) for token in tokens]
        sequences.append(seq)
    
    return sequences

In [42]:
PAD_IDX = word_to_idx["<PAD>"]
UNK_IDX = word_to_idx["<UNK>"]

In [43]:
X_train_idx = texts_to_indices(X_train, word_to_idx, UNK_IDX)
X_test_idx  = texts_to_indices(X_test, word_to_idx, UNK_IDX)

In [44]:
max_len = int(np.percentile(train_lengths, 99.5))
print("max_len =", max_len)

max_len = 12507


In [45]:
X_train_pad = pad_sequences(X_train_idx, max_len=max_len, pad_value=PAD_IDX)
X_test_pad  = pad_sequences(X_test_idx, max_len=max_len, pad_value=PAD_IDX)

In [46]:
print("X_train_pad.shape =", X_train_pad.shape)
print("X_test_pad.shape =", X_test_pad.shape)

X_train_pad.shape = (1079, 12507)
X_test_pad.shape = (717, 12507)


In [47]:
import numpy as np
y_train = np.array(y_train)
y_test = np.array(y_test)

print(y_train.shape, y_test.shape)

(1079,) (717,)


Bidirectional LSTM

In [48]:
import torch
import torch.nn as nn
PAD_IDX = word_to_idx["<PAD>"]

X_train_tensor = torch.tensor(X_train_pad, dtype=torch.long)
X_test_tensor  = torch.tensor(X_test_pad, dtype=torch.long)

y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor  = torch.tensor(y_test, dtype=torch.long)

train_lengths = (X_train_tensor != PAD_IDX).sum(dim=1)
test_lengths  = (X_test_tensor != PAD_IDX).sum(dim=1)

In [49]:
train_mask = train_lengths > 0
test_mask = test_lengths > 0

X_train_tensor = X_train_tensor[train_mask]
y_train_tensor = y_train_tensor[train_mask]
train_lengths = train_lengths[train_mask]

X_test_tensor = X_test_tensor[test_mask]
y_test_tensor = y_test_tensor[test_mask]
test_lengths = test_lengths[test_mask]

In [50]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, train_lengths, y_train_tensor)
test_dataset  = TensorDataset(X_test_tensor, test_lengths, y_test_tensor)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [51]:


class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, pad_idx, num_classes=2):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_idx
        )

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        # 只拼接 h_forward 和 h_backward
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

        self.init_forget_gate_bias(value=1.0)

    def init_forget_gate_bias(self, value=1.0):
        hidden_dim = self.lstm.hidden_size
        for name, param in self.lstm.named_parameters():
            if "bias" in name:
                param.data[hidden_dim:2 * hidden_dim].fill_(value)

    def forward(self, x, lengths):
        # x: [B, T]
        # lengths: [B]
        x_embed = self.embedding(x)   # [B, T, E]

        packed = nn.utils.rnn.pack_padded_sequence(
            x_embed, lengths.cpu(), batch_first=True, enforce_sorted=False
        )

        _, (h_n, c_n) = self.lstm(packed)

        # 单层双向LSTM时:
        # h_n.shape = [2, B, H]
        h_forward = h_n[0]   # [B, H]
        h_backward = h_n[1]  # [B, H]

        h_concat = torch.cat([h_forward, h_backward], dim=1)  # [B, 2H]
        logits = self.fc(h_concat)  # [B, 2]

        return logits

In [52]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BiLSTMClassifier(
    vocab_size=len(word_to_idx),
    embed_dim=512,
    hidden_dim=512,
    pad_idx=PAD_IDX,
    num_classes=2
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [53]:
from sklearn.metrics import accuracy_score

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []

    for X_batch, lengths_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)
        lengths_batch = lengths_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = model(X_batch, lengths_batch)   # [B, 2]
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(y_batch.detach().cpu().numpy())

    return total_loss / len(dataloader), accuracy_score(all_labels, all_preds)

In [54]:
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X_batch, lengths_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)
            lengths_batch = lengths_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch, lengths_batch)
            loss = criterion(logits, y_batch)

            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    return total_loss / len(dataloader), accuracy_score(all_labels, all_preds), all_preds, all_labels

In [55]:
epochs = 20

for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc, _, _ = evaluate(model, test_loader, criterion, device)

    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
          f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

Epoch [1/20] Train Loss: 0.6425, Train Acc: 0.6493 | Test Loss: 0.6014, Test Acc: 0.6590
Epoch [2/20] Train Loss: 0.2057, Train Acc: 0.9423 | Test Loss: 0.6683, Test Acc: 0.7108
Epoch [3/20] Train Loss: 0.0432, Train Acc: 0.9905 | Test Loss: 0.7417, Test Acc: 0.7151
Epoch [4/20] Train Loss: 0.0045, Train Acc: 1.0000 | Test Loss: 1.0270, Test Acc: 0.7151
Epoch [5/20] Train Loss: 0.0015, Train Acc: 1.0000 | Test Loss: 0.9971, Test Acc: 0.7165
Epoch [6/20] Train Loss: 0.0006, Train Acc: 1.0000 | Test Loss: 1.0818, Test Acc: 0.7194
Epoch [7/20] Train Loss: 0.0003, Train Acc: 1.0000 | Test Loss: 1.1303, Test Acc: 0.7266
Epoch [8/20] Train Loss: 0.0002, Train Acc: 1.0000 | Test Loss: 1.1614, Test Acc: 0.7252
Epoch [9/20] Train Loss: 0.0002, Train Acc: 1.0000 | Test Loss: 1.1955, Test Acc: 0.7295
Epoch [10/20] Train Loss: 0.0002, Train Acc: 1.0000 | Test Loss: 1.2189, Test Acc: 0.7281
Epoch [11/20] Train Loss: 0.0001, Train Acc: 1.0000 | Test Loss: 1.2359, Test Acc: 0.7281
Epoch [12/20] Train

In [56]:
from sklearn.metrics import classification_report

test_loss, test_acc, y_pred, y_true = evaluate(model, test_loader, criterion, device)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_acc)
print(classification_report(y_true, y_pred, digits=4))

Test Loss: 1.4110268977555362
Test Accuracy: 0.7237410071942446
              precision    recall  f1-score   support

           0     0.7409    0.5884    0.6559       311
           1     0.7143    0.8333    0.7692       384

    accuracy                         0.7237       695
   macro avg     0.7276    0.7109    0.7126       695
weighted avg     0.7262    0.7237    0.7185       695



In [57]:
save_path = "bilstm_text_classifier.pth"

checkpoint = {
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "word_to_idx": word_to_idx,
    "vocab_size": len(word_to_idx),
    "embed_dim": 512,
    "hidden_dim": 512,
    "pad_idx": PAD_IDX,
    "num_classes": 2,
    "max_len": X_train_pad.shape[1],
    "test_acc": test_acc
}

torch.save(checkpoint, save_path)
print(f"模型已保存到: {save_path}")

模型已保存到: bilstm_text_classifier.pth


Bidirectional RNN

In [81]:
X_train_tensor = torch.tensor(X_train_pad, dtype=torch.long)
X_test_tensor  = torch.tensor(X_test_pad, dtype=torch.long)

# CrossEntropyLoss 要求标签是 LongTensor，且类别编号是 0/1
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor  = torch.tensor(y_test, dtype=torch.long)


In [196]:
batch_size = 128

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [197]:
class BiRNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, pad_idx, num_classes=2):
        super().__init__()
        
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_idx
        )
        
        self.rnn = nn.RNN(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
            nonlinearity='tanh'
        )
        
        # 双向，所以维度是 hidden_dim * 2
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        # x: [batch_size, seq_len]
        x_embed = self.embedding(x)   # [batch_size, seq_len, embed_dim]
        
        output, h_n = self.rnn(x_embed)
        # output: [batch_size, seq_len, hidden_dim * 2]
        # h_n:    [num_layers * num_directions, batch_size, hidden_dim]
        # 单层双向时: [2, batch_size, hidden_dim]

        h_forward = h_n[0]   # [batch_size, hidden_dim]
        h_backward = h_n[1]  # [batch_size, hidden_dim]

        h_concat = torch.cat([h_forward, h_backward], dim=1)  # [batch_size, hidden_dim * 2]
        logits = self.fc(h_concat)                            # [batch_size, 2]
        
        return logits

In [198]:

import torch
import torch.nn as nn

from sklearn.metrics import accuracy_score, classification_report
vocab_size = len(word_to_idx)
PAD_IDX = word_to_idx["<PAD>"]
embed_dim = 128
hidden_dim = 256

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BiRNNClassifier(
    vocab_size=vocab_size,
    embed_dim=embed_dim,
    hidden_dim=hidden_dim,
    pad_idx=PAD_IDX,
    num_classes=2
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# ========= 3. 训练/评估 =========
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(y_batch.detach().cpu().numpy())
    
    return total_loss / len(dataloader), accuracy_score(all_labels, all_preds)

def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            total_loss += loss.item()
            
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
    
    return total_loss / len(dataloader), accuracy_score(all_labels, all_preds), all_preds, all_labels

# ========= 4. 训练 =========
epochs = 10

In [207]:

for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc, _, _ = evaluate(model, test_loader, criterion, device)
    
    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
          f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

# ========= 5. 最终测试 =========
test_loss, test_acc, y_pred, y_true = evaluate(model, test_loader, criterion, device)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_acc)
print(classification_report(y_true, y_pred, digits=4))

OutOfMemoryError: CUDA out of memory. Tried to allocate 53.29 GiB. GPU 0 has a total capacity of 47.54 GiB of which 18.70 GiB is free. Process 3338584 has 838.00 MiB memory in use. Process 3344161 has 1.09 GiB memory in use. Process 3351175 has 1.14 GiB memory in use. Process 3865861 has 4.12 GiB memory in use. Including non-PyTorch memory, this process has 21.65 GiB memory in use. Of the allocated memory 20.53 GiB is allocated by PyTorch, and 812.91 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)